# 01 – Exploratory Data Analysis

> The bundled dataset is **synthetic**; `outage_event` is a *simulated reliability-risk event*, not a real outage. Re-run this notebook on real data and replace the observations below with what the data actually shows.

In [ ]:
import sys; sys.path.insert(0, '..')
import os; os.chdir('..')   # run from repo root
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src.preprocessing import load_raw, clean
from src.features import build_features, TARGET, FEATURES

raw = load_raw('data/raw/grid_data.csv')
raw.head()

## Data quality checks

In [ ]:
print('Rows:', len(raw), '| range:', raw.timestamp.min(), '->', raw.timestamp.max())
print('Duplicate timestamps:', raw.timestamp.duplicated().sum())
print('Missing values per column:'); print(raw.isna().sum())
print('Negative demand/generation rows:', (raw[['demand_mw','solar_mw','wind_mw']] < 0).any(axis=1).sum())
raw.describe().T

In [ ]:
df = build_features(clean(raw))
print('Event rate (next interval):', df[TARGET].mean().round(4))

## Demand, renewables, temperature

In [ ]:
w = df.iloc[:24*14]   # first two weeks
fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
ax[0].plot(w.timestamp, w.demand_mw); ax[0].set_ylabel('Demand (MW)')
ax[1].plot(w.timestamp, w.solar_mw, label='Solar'); ax[1].plot(w.timestamp, w.wind_mw, label='Wind'); ax[1].legend(); ax[1].set_ylabel('Generation (MW)')
ax[2].plot(w.timestamp, w.net_load, color='purple'); ax[2].set_ylabel('Net load (MW)')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(df.temperature, df.demand_mw, s=2, alpha=.3); ax[0].set_xlabel('Temperature (°C)'); ax[0].set_ylabel('Demand (MW)'); ax[0].set_title('Demand vs temperature')
ax[1].scatter(df.renewable_mw, df.demand_mw, s=2, alpha=.3); ax[1].set_xlabel('Renewables (MW)'); ax[1].set_title('Renewables vs demand')
plt.tight_layout(); plt.show()

## Reliability events

In [ ]:
m = df.set_index('timestamp')[TARGET].resample('ME').sum()
m.plot(kind='bar', figsize=(12, 3), title='Reliability events per month'); plt.show()

print('Event rate by peak-load flag:'); print(df.groupby('peak_load')[TARGET].mean())
df['ramp_bucket'] = pd.qcut(df.renewable_ramp, 5, duplicates='drop')
print('Event rate by renewable-ramp quintile:'); print(df.groupby('ramp_bucket', observed=True)[TARGET].mean())
print('Event rate by events in last 24h:'); print(df.groupby(df.outages_24h.clip(upper=4))[TARGET].mean())

## Correlation matrix

In [ ]:
corr = df[FEATURES + [TARGET]].corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
plt.colorbar(im); plt.tight_layout(); plt.show()
corr[TARGET].drop(TARGET).sort_values(ascending=False)

## Observations (fill in from YOUR data)

- Is the event rate higher in peak-load hours? (see table above)
- Are sharp negative renewable ramps associated with more events?
- Do events cluster in time (autocorrelation → `outages_24h` should matter)?
- Any seasonality in demand/temperature that the time-aware split must respect?